# W05 · Grid-PEPS on images (Table 1) / 影像上的 Grid-PEPS(Table 1)

**English.** Train a grid baseline and Grid-PEPS on a Kodak image and compare
PSNR at matched parameter budgets — the G-PEPS rows of paper Table 1. We also
draw the dual-scatter comparison (paper Fig. 7 style).

**繁體中文.** 在 Kodak 影像上訓練 grid baseline 與 Grid-PEPS,在相同參數預算下
比較 PSNR —— 即論文 Table 1 的 G-PEPS 列;並畫雙散點對比(Fig.7 風格)。

In [1]:
import sys, os; sys.path.insert(0, os.path.abspath('..'))
import torch, matplotlib.pyplot as plt
from peps.train import auto_device
device = auto_device(); print('device', device)

device cuda


/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


In [2]:
from apps.image.data import load_image, image_to_coords_targets, find_kodak
from apps.image.build import build_grid, build_grid_peps
from peps.train import fit, TrainConfig, render_full
from peps.metrics import psnr, ssim
img = load_image(find_kodak(1), max_size=384)
coords, targets, (H, W) = image_to_coords_targets(img)
print('image', H, W)

image 256 384


## 1. Train grid vs Grid-PEPS at matched budget / 相同預算下對比

In [3]:
def train_eval(builder_out, steps=2500):
    model, pc = builder_out
    fit(model, coords, targets,
        TrainConfig(steps=steps, batch_size=32768, lr=1e-2, device=device))
    pred = render_full(model, coords, device=device).reshape(H, W, 3).clamp(0, 1)
    return pc, psnr(pred, img), ssim(pred, img), pred

rows = {}
rows['grid']      = train_eval(build_grid(resolution=128, feature_dim=8))
rows['grid_peps'] = train_eval(build_grid_peps(resolution=128, feature_dim=8, num_frequencies=6))
for k, (pc, ps, ss, _) in rows.items():
    print(f'{k:10s} params={pc:8d}  PSNR={ps:6.2f} dB  SSIM={ss:.4f}')

grid       params=  136003  PSNR= 37.74 dB  SSIM=0.9966
grid_peps  params=  142147  PSNR= 42.20 dB  SSIM=0.9988


## 2. Visual + dual-scatter (Fig. 7 style) / 視覺與雙散點

In [4]:
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(img); ax[0].set_title('target'); ax[0].axis('off')
ax[1].imshow(rows['grid'][3]);      ax[1].set_title(f"grid {rows['grid'][1]:.1f} dB"); ax[1].axis('off')
ax[2].imshow(rows['grid_peps'][3]); ax[2].set_title(f"Grid-PEPS {rows['grid_peps'][1]:.1f} dB"); ax[2].axis('off')
plt.show()

## 3. Save Table 1 row / 存 Table 1 列
Results are appended to `results/table1_image.csv` for the W07 wrap-up.

In [5]:
import csv, os
os.makedirs('../results', exist_ok=True)
path = '../results/table1_image.csv'
new = not os.path.exists(path)
with open(path, 'a', newline='') as f:
    w = csv.writer(f)
    if new: w.writerow(['method', 'params', 'psnr', 'ssim'])
    for k, (pc, ps, ss, _) in rows.items():
        w.writerow([k, pc, round(ps, 3), round(ss, 4)])
print('saved', path)

saved ../results/table1_image.csv
